In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate
!pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow

############################################################################################################
########## It will be asked for your huggingface access token to load qwen packages ########################
############################################################################################################



train dataset

In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

# MODEL_NAME = "google/gemma-3-4b-it"
# MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
MODEL_NAME = "models--Qwen--Qwen2.5-VL-3B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "India"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/train/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/train/train.csv" # MDMD dataset CSV
OUTPUT_DIR = "Qwen2.5-VL-3B-Instruct/results_train_Tamil_India/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

/home/wangkongqiang/miniconda3/envs/CC-MMD/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 824/824 [00:29<00:00, 27.77it/s]


Model ready.



Classifying:  10%|██████▌                                                          | 115/1137 [03:33<32:06,  1.89s/meme]

  [WARN] JSON parse failed for 363: Expecting ',' delimiter: line 3 column 239 (char 254)
  Raw response: ```json
{
  "label": 1,
  "reason": "The image shows a man performing a martial arts move in front of a group of people, which could be interpreted as a display of strength or dominance. The context of the meme, which includes references to a fight scene and a "dupe interview," suggests a focus on physical prowess and possibly gendered expectations of masculinity."
}
```


Classifying: 100%|████████████████████████████████████████████████████████████████| 1137/1137 [35:03<00:00,  1.85s/meme]


Results saved → results/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.json  |  results/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.txt

Summary:  Total=1137  Misogyny=1135  Not-misogyny=2  Errors=0


test dataset

In [2]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

# MODEL_NAME = "google/gemma-3-4b-it"
# MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
MODEL_NAME = "models--Qwen--Qwen2.5-VL-3B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "India"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Qwen2.5-VL-3B-Instruct/results_test_Tamil_India/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 824/824 [00:27<00:00, 30.25it/s]


Model ready.



Classifying:  97%|███████████████████████████████████████████████████████████████▊  | 344/356 [10:34<00:57,  4.83s/meme]

  [WARN] JSON parse failed for 1028: Unterminated string starting at: line 3 column 13 (char 28)
  Raw response: ```json
{
  "label": 1,
  "reason": "The meme text in Tamil translates to 'I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a 

Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [10:57<00:00,  1.85s/meme]


Results saved → Qwen2.5-VL-3B-Instruct/results_test_Tamil_India/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.json  |  Qwen2.5-VL-3B-Instruct/results_test_Tamil_India/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.txt

Summary:  Total=356  Misogyny=355  Not-misogyny=1  Errors=0


In [3]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

# MODEL_NAME = "google/gemma-3-4b-it"
# MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
MODEL_NAME = "models--Qwen--Qwen2.5-VL-3B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "China"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Qwen2.5-VL-3B-Instruct/results_test_Tamil_China/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 824/824 [00:27<00:00, 29.84it/s]


Model ready.



Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [11:26<00:00,  1.93s/meme]


Results saved → Qwen2.5-VL-3B-Instruct/results_test_Tamil_China/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.json  |  Qwen2.5-VL-3B-Instruct/results_test_Tamil_China/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.txt

Summary:  Total=356  Misogyny=356  Not-misogyny=0  Errors=0


In [4]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

# MODEL_NAME = "google/gemma-3-4b-it"
# MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
MODEL_NAME = "models--Qwen--Qwen2.5-VL-3B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "Ireland"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Qwen2.5-VL-3B-Instruct/results_test_Tamil_Ireland/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 824/824 [00:26<00:00, 30.88it/s]


Model ready.



Classifying:  97%|███████████████████████████████████████████████████████████████▊  | 344/356 [11:05<01:06,  5.53s/meme]

  [WARN] JSON parse failed for 1028: Unterminated string starting at: line 3 column 13 (char 28)
  Raw response: ```json
{
  "label": 1,
  "reason": "The meme text in Tamil translates to 'I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a woman, I'm a 

Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [11:26<00:00,  1.93s/meme]


Results saved → Qwen2.5-VL-3B-Instruct/results_test_Tamil_Ireland/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.json  |  Qwen2.5-VL-3B-Instruct/results_test_Tamil_Ireland/models--Qwen--Qwen2.5-VL-3B-Instruct_tamil.txt

Summary:  Total=356  Misogyny=355  Not-misogyny=1  Errors=0


test dataset

In [5]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

# MODEL_NAME = "google/gemma-3-4b-it"
# MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "India"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Qwen2.5-VL-7B-Instruct/results_test_Tamil_India/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 729/729 [00:50<00:00, 14.30it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


Model ready.



Classifying: 100%|████████████████████████████████████████████████████████████████| 356/356 [2:49:52<00:00, 28.63s/meme]


Results saved → Qwen2.5-VL-7B-Instruct/results_test_Tamil_India/models--Qwen--Qwen2.5-VL-7B-Instruct_tamil.json  |  Qwen2.5-VL-7B-Instruct/results_test_Tamil_India/models--Qwen--Qwen2.5-VL-7B-Instruct_tamil.txt

Summary:  Total=356  Misogyny=336  Not-misogyny=20  Errors=0


In [6]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

# MODEL_NAME = "google/gemma-3-4b-it"
# MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "China"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Qwen2.5-VL-7B-Instruct/results_test_Tamil_China/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 729/729 [00:27<00:00, 26.10it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


Model ready.



Classifying: 100%|████████████████████████████████████████████████████████████████| 356/356 [7:14:50<00:00, 73.29s/meme]


Results saved → Qwen2.5-VL-7B-Instruct/results_test_Tamil_China/models--Qwen--Qwen2.5-VL-7B-Instruct_tamil.json  |  Qwen2.5-VL-7B-Instruct/results_test_Tamil_China/models--Qwen--Qwen2.5-VL-7B-Instruct_tamil.txt

Summary:  Total=356  Misogyny=283  Not-misogyny=73  Errors=0


In [7]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------

# MODEL_NAME = "google/gemma-3-4b-it"
# MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "Ireland"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Qwen2.5-VL-7B-Instruct/results_test_Tamil_Ireland/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
Reason over BOTH the image and the transcription text together.

Important Notes:
- Be objective and precise.
- Do not interpret humor or irony as neutral if it carries misogynistic meaning.

Respond ONLY with this JSON (no markdown, no extra text):
{{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
label: 1 = misogyny, 0 = not-misogyny"""

# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------

def classify_image(image_path, transcription, country, language):
    """
    Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

    Args:
        image_path   : path to the meme image
        transcription: OCR-extracted text from the meme
        country      : cultural context (e.g. "India, China, Ireland")
        language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

    Returns:
        dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
    """
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = PROMPT.format(language=language, country=country)

    # Append OCR transcription to the prompt
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
            {"type": "text",  "text": user_message},
        ],
    }]

    # Prepare inputs
    text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(DEVICE)

    # Generate
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=512,
                                 do_sample=False, temperature=None, top_p=None)

    result = processor.batch_decode(
        [out_ids[0][len(inputs.input_ids[0]):]],
        skip_special_tokens=True,
    )[0].strip()

    # Parse JSON response
    try:
        if "```json" in result:
            json_str = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            json_str = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            json_str = result[start:end] if start >= 0 and end > start else result

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {image_id}: {e}")
        print(f"  Raw response: {result}")
        classification = "misogyny" if ("misogyny" in result.lower()
                                        and "not-misogyny" not in result.lower()
                                        and "not misogyny" not in result.lower()) else "not-misogyny"
        explanation = result or "No response."

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 729/729 [00:34<00:00, 21.01it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


Model ready.



Classifying: 100%|████████████████████████████████████████████████████████████████| 356/356 [3:10:19<00:00, 32.08s/meme]


Results saved → Qwen2.5-VL-7B-Instruct/results_test_Tamil_Ireland/models--Qwen--Qwen2.5-VL-7B-Instruct_tamil.json  |  Qwen2.5-VL-7B-Instruct/results_test_Tamil_Ireland/models--Qwen--Qwen2.5-VL-7B-Instruct_tamil.txt

Summary:  Total=356  Misogyny=307  Not-misogyny=49  Errors=0


test dataset

✅ 二、修改后的完整关键代码（可直接替换）

In [9]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
# from PIL import Image
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import AutoTokenizer, AutoModelForCausalLM
# from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------
# 🔧 1. 模型加载（替换整段）
# MODEL_NAME = "google/gemma-3-4b-it"
MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
# MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "India"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Gemma-3-4B-it-abliterated/results_test_Tamil_India/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
#     device_map="auto",
# )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
# 🔧 2. 删除这些内容（全部删掉）
# processor = AutoProcessor.from_pretrained(
#     MODEL_NAME,
#     min_pixels=256 * 28 * 28,
#     max_pixels=1280 * 28 * 28,
# )
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
# ✅ 改 prompt（很重要）
# PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
# Reason over BOTH the image and the transcription text together.

# Important Notes:
# - Be objective and precise.
# - Do not interpret humor or irony as neutral if it carries misogynistic meaning.

# Respond ONLY with this JSON (no markdown, no extra text):
# {{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
# label: 1 = misogyny, 0 = not-misogyny"""
PROMPT = """Given a {language} meme OCR transcription, classify whether it is misogynistic (1) or not (0) in {country} cultural context.

Note:
- You DO NOT see the image.
- Base your judgment only on the text.
- Be conservative: only label misogyny if clearly offensive toward women.

Respond ONLY JSON:
{{"label": 1, "reason": "..."}}
"""
# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------
# 🔧 3. 修改 classify_image（⚠️最重要）

# 👉 改成 纯文本推理
# def classify_image(image_path, transcription, country, language):
#     """
#     Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

#     Args:
#         image_path   : path to the meme image
#         transcription: OCR-extracted text from the meme
#         country      : cultural context (e.g. "India, China, Ireland")
#         language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

#     Returns:
#         dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
#     """
#     image_id = os.path.splitext(os.path.basename(image_path))[0]
#     prompt   = PROMPT.format(language=language, country=country)

#     # Append OCR transcription to the prompt
#     user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

#     messages = [{
#         "role": "user",
#         "content": [
#             {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
#             {"type": "text",  "text": user_message},
#         ],
#     }]

#     # Prepare inputs
#     text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
#                              padding=True, return_tensors="pt").to(DEVICE)

#     # Generate
#     with torch.no_grad():
#         out_ids = model.generate(**inputs, max_new_tokens=512,
#                                  do_sample=False, temperature=None, top_p=None)

#     result = processor.batch_decode(
#         [out_ids[0][len(inputs.input_ids[0]):]],
#         skip_special_tokens=True,
#     )[0].strip()

#     # Parse JSON response
#     try:
#         if "```json" in result:
#             json_str = result.split("```json")[1].split("```")[0].strip()
#         elif "```" in result:
#             json_str = result.split("```")[1].strip()
#         else:
#             start = result.find("{")
#             end   = result.rfind("}") + 1
#             json_str = result[start:end] if start >= 0 and end > start else result

#         parsed         = json.loads(json_str)
#         label          = int(parsed.get("label", 0))
#         classification = "misogyny" if label == 1 else "not-misogyny"
#         explanation    = parsed.get("reason", "")

#     except Exception as e:
#         print(f"  [WARN] JSON parse failed for {image_id}: {e}")
#         print(f"  Raw response: {result}")
#         classification = "misogyny" if ("misogyny" in result.lower()
#                                         and "not-misogyny" not in result.lower()
#                                         and "not misogyny" not in result.lower()) else "not-misogyny"
#         explanation = result or "No response."

#     return {
#         "image_id":       image_id,
#         "classification": classification,
#         "explanation":    explanation,
#         "full_response":  result,
#     }
def classify_image(image_path, transcription, country, language):

    image_id = os.path.splitext(os.path.basename(image_path))[0]

    prompt = PROMPT.format(language=language, country=country)

    # ❗ 不再使用图片
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    inputs = tokenizer(
        user_message,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    result = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # ---- JSON解析（保持你原来的）----
    try:
        start = result.find("{")
        end   = result.rfind("}") + 1
        json_str = result[start:end]

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"[WARN] JSON parse failed: {e}")
        classification = "not-misogyny"
        explanation = result

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 883/883 [00:32<00:00, 26.96it/s]


Model ready.



Classifying:   2%|█▏                                                                  | 6/356 [00:09<07:19,  1.26s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


...

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying:  96%|███████████████████████████████████████████████████████████████▍  | 342/356 [09:17<00:23,  1.67s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [09:42<00:00,  1.64s/meme]


Results saved → Gemma-3-4B-it-abliterated/results_test_Tamil_India/models--mlabonne--gemma-3-4b-it-abliterated_tamil.json  |  Gemma-3-4B-it-abliterated/results_test_Tamil_India/models--mlabonne--gemma-3-4b-it-abliterated_tamil.txt

Summary:  Total=356  Misogyny=119  Not-misogyny=237  Errors=0


In [10]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
# from PIL import Image
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import AutoTokenizer, AutoModelForCausalLM
# from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------
# 🔧 1. 模型加载（替换整段）
# MODEL_NAME = "google/gemma-3-4b-it"
MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
# MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "China"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Gemma-3-4B-it-abliterated/results_test_Tamil_China/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
#     device_map="auto",
# )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
# 🔧 2. 删除这些内容（全部删掉）
# processor = AutoProcessor.from_pretrained(
#     MODEL_NAME,
#     min_pixels=256 * 28 * 28,
#     max_pixels=1280 * 28 * 28,
# )
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
# ✅ 改 prompt（很重要）
# PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
# Reason over BOTH the image and the transcription text together.

# Important Notes:
# - Be objective and precise.
# - Do not interpret humor or irony as neutral if it carries misogynistic meaning.

# Respond ONLY with this JSON (no markdown, no extra text):
# {{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
# label: 1 = misogyny, 0 = not-misogyny"""
PROMPT = """Given a {language} meme OCR transcription, classify whether it is misogynistic (1) or not (0) in {country} cultural context.

Note:
- You DO NOT see the image.
- Base your judgment only on the text.
- Be conservative: only label misogyny if clearly offensive toward women.

Respond ONLY JSON:
{{"label": 1, "reason": "..."}}
"""
# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------
# 🔧 3. 修改 classify_image（⚠️最重要）

# 👉 改成 纯文本推理
# def classify_image(image_path, transcription, country, language):
#     """
#     Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

#     Args:
#         image_path   : path to the meme image
#         transcription: OCR-extracted text from the meme
#         country      : cultural context (e.g. "India, China, Ireland")
#         language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

#     Returns:
#         dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
#     """
#     image_id = os.path.splitext(os.path.basename(image_path))[0]
#     prompt   = PROMPT.format(language=language, country=country)

#     # Append OCR transcription to the prompt
#     user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

#     messages = [{
#         "role": "user",
#         "content": [
#             {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
#             {"type": "text",  "text": user_message},
#         ],
#     }]

#     # Prepare inputs
#     text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
#                              padding=True, return_tensors="pt").to(DEVICE)

#     # Generate
#     with torch.no_grad():
#         out_ids = model.generate(**inputs, max_new_tokens=512,
#                                  do_sample=False, temperature=None, top_p=None)

#     result = processor.batch_decode(
#         [out_ids[0][len(inputs.input_ids[0]):]],
#         skip_special_tokens=True,
#     )[0].strip()

#     # Parse JSON response
#     try:
#         if "```json" in result:
#             json_str = result.split("```json")[1].split("```")[0].strip()
#         elif "```" in result:
#             json_str = result.split("```")[1].strip()
#         else:
#             start = result.find("{")
#             end   = result.rfind("}") + 1
#             json_str = result[start:end] if start >= 0 and end > start else result

#         parsed         = json.loads(json_str)
#         label          = int(parsed.get("label", 0))
#         classification = "misogyny" if label == 1 else "not-misogyny"
#         explanation    = parsed.get("reason", "")

#     except Exception as e:
#         print(f"  [WARN] JSON parse failed for {image_id}: {e}")
#         print(f"  Raw response: {result}")
#         classification = "misogyny" if ("misogyny" in result.lower()
#                                         and "not-misogyny" not in result.lower()
#                                         and "not misogyny" not in result.lower()) else "not-misogyny"
#         explanation = result or "No response."

#     return {
#         "image_id":       image_id,
#         "classification": classification,
#         "explanation":    explanation,
#         "full_response":  result,
#     }
def classify_image(image_path, transcription, country, language):

    image_id = os.path.splitext(os.path.basename(image_path))[0]

    prompt = PROMPT.format(language=language, country=country)

    # ❗ 不再使用图片
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    inputs = tokenizer(
        user_message,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    result = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # ---- JSON解析（保持你原来的）----
    try:
        start = result.find("{")
        end   = result.rfind("}") + 1
        json_str = result[start:end]

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"[WARN] JSON parse failed: {e}")
        classification = "not-misogyny"
        explanation = result

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 883/883 [00:33<00:00, 26.67it/s]


Model ready.



Classifying:   0%|▏                                                                   | 1/356 [00:01<09:52,  1.67s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


...

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying:  99%|█████████████████████████████████████████████████████████████████ | 351/356 [08:37<00:10,  2.14s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [08:47<00:00,  1.48s/meme]


Results saved → Gemma-3-4B-it-abliterated/results_test_Tamil_China/models--mlabonne--gemma-3-4b-it-abliterated_tamil.json  |  Gemma-3-4B-it-abliterated/results_test_Tamil_China/models--mlabonne--gemma-3-4b-it-abliterated_tamil.txt

Summary:  Total=356  Misogyny=87  Not-misogyny=269  Errors=0


In [11]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
# from PIL import Image
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import AutoTokenizer, AutoModelForCausalLM
# from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------
# 🔧 1. 模型加载（替换整段）
# MODEL_NAME = "google/gemma-3-4b-it"
MODEL_NAME = "models--mlabonne--gemma-3-4b-it-abliterated"
# MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "Ireland"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Gemma-3-4B-it-abliterated/results_test_Tamil_Ireland/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
#     device_map="auto",
# )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
# 🔧 2. 删除这些内容（全部删掉）
# processor = AutoProcessor.from_pretrained(
#     MODEL_NAME,
#     min_pixels=256 * 28 * 28,
#     max_pixels=1280 * 28 * 28,
# )
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
# ✅ 改 prompt（很重要）
# PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
# Reason over BOTH the image and the transcription text together.

# Important Notes:
# - Be objective and precise.
# - Do not interpret humor or irony as neutral if it carries misogynistic meaning.

# Respond ONLY with this JSON (no markdown, no extra text):
# {{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
# label: 1 = misogyny, 0 = not-misogyny"""
PROMPT = """Given a {language} meme OCR transcription, classify whether it is misogynistic (1) or not (0) in {country} cultural context.

Note:
- You DO NOT see the image.
- Base your judgment only on the text.
- Be conservative: only label misogyny if clearly offensive toward women.

Respond ONLY JSON:
{{"label": 1, "reason": "..."}}
"""
# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------
# 🔧 3. 修改 classify_image（⚠️最重要）

# 👉 改成 纯文本推理
# def classify_image(image_path, transcription, country, language):
#     """
#     Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

#     Args:
#         image_path   : path to the meme image
#         transcription: OCR-extracted text from the meme
#         country      : cultural context (e.g. "India, China, Ireland")
#         language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

#     Returns:
#         dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
#     """
#     image_id = os.path.splitext(os.path.basename(image_path))[0]
#     prompt   = PROMPT.format(language=language, country=country)

#     # Append OCR transcription to the prompt
#     user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

#     messages = [{
#         "role": "user",
#         "content": [
#             {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
#             {"type": "text",  "text": user_message},
#         ],
#     }]

#     # Prepare inputs
#     text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
#                              padding=True, return_tensors="pt").to(DEVICE)

#     # Generate
#     with torch.no_grad():
#         out_ids = model.generate(**inputs, max_new_tokens=512,
#                                  do_sample=False, temperature=None, top_p=None)

#     result = processor.batch_decode(
#         [out_ids[0][len(inputs.input_ids[0]):]],
#         skip_special_tokens=True,
#     )[0].strip()

#     # Parse JSON response
#     try:
#         if "```json" in result:
#             json_str = result.split("```json")[1].split("```")[0].strip()
#         elif "```" in result:
#             json_str = result.split("```")[1].strip()
#         else:
#             start = result.find("{")
#             end   = result.rfind("}") + 1
#             json_str = result[start:end] if start >= 0 and end > start else result

#         parsed         = json.loads(json_str)
#         label          = int(parsed.get("label", 0))
#         classification = "misogyny" if label == 1 else "not-misogyny"
#         explanation    = parsed.get("reason", "")

#     except Exception as e:
#         print(f"  [WARN] JSON parse failed for {image_id}: {e}")
#         print(f"  Raw response: {result}")
#         classification = "misogyny" if ("misogyny" in result.lower()
#                                         and "not-misogyny" not in result.lower()
#                                         and "not misogyny" not in result.lower()) else "not-misogyny"
#         explanation = result or "No response."

#     return {
#         "image_id":       image_id,
#         "classification": classification,
#         "explanation":    explanation,
#         "full_response":  result,
#     }
def classify_image(image_path, transcription, country, language):

    image_id = os.path.splitext(os.path.basename(image_path))[0]

    prompt = PROMPT.format(language=language, country=country)

    # ❗ 不再使用图片
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    inputs = tokenizer(
        user_message,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    result = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # ---- JSON解析（保持你原来的）----
    try:
        start = result.find("{")
        end   = result.rfind("}") + 1
        json_str = result[start:end]

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"[WARN] JSON parse failed: {e}")
        classification = "not-misogyny"
        explanation = result

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 883/883 [00:31<00:00, 28.20it/s]


Model ready.



Classifying:   1%|▉                                                                   | 5/356 [00:07<09:31,  1.63s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


...

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying:  99%|█████████████████████████████████████████████████████████████████▋| 354/356 [08:56<00:03,  1.88s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [08:59<00:00,  1.51s/meme]


Results saved → Gemma-3-4B-it-abliterated/results_test_Tamil_Ireland/models--mlabonne--gemma-3-4b-it-abliterated_tamil.json  |  Gemma-3-4B-it-abliterated/results_test_Tamil_Ireland/models--mlabonne--gemma-3-4b-it-abliterated_tamil.txt

Summary:  Total=356  Misogyny=80  Not-misogyny=276  Errors=0


test dataset

你这个代码现在是专门为 多模态模型（Qwen2.5-VL） 写的，而 models--mlabonne--gemma-3-1b-it-abliterated 本质是 纯文本 LLM（Gemma-3） —— ❗不支持图像输入。

✅ 二、修改后的完整关键代码（可直接替换）

In [12]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
# from PIL import Image
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import AutoTokenizer, AutoModelForCausalLM
# from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------
# 🔧 1. 模型加载（替换整段）
# MODEL_NAME = "google/gemma-3-4b-it"
MODEL_NAME = "models--mlabonne--gemma-3-1b-it-abliterated"
# MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "India"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Gemma-3-1B-it-abliterated/results_test_Tamil_India/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
#     device_map="auto",
# )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
# 🔧 2. 删除这些内容（全部删掉）
# processor = AutoProcessor.from_pretrained(
#     MODEL_NAME,
#     min_pixels=256 * 28 * 28,
#     max_pixels=1280 * 28 * 28,
# )
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
# ✅ 改 prompt（很重要）
# PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
# Reason over BOTH the image and the transcription text together.

# Important Notes:
# - Be objective and precise.
# - Do not interpret humor or irony as neutral if it carries misogynistic meaning.

# Respond ONLY with this JSON (no markdown, no extra text):
# {{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
# label: 1 = misogyny, 0 = not-misogyny"""
PROMPT = """Given a {language} meme OCR transcription, classify whether it is misogynistic (1) or not (0) in {country} cultural context.

Note:
- You DO NOT see the image.
- Base your judgment only on the text.
- Be conservative: only label misogyny if clearly offensive toward women.

Respond ONLY JSON:
{{"label": 1, "reason": "..."}}
"""
# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------
# 🔧 3. 修改 classify_image（⚠️最重要）

# 👉 改成 纯文本推理
# def classify_image(image_path, transcription, country, language):
#     """
#     Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

#     Args:
#         image_path   : path to the meme image
#         transcription: OCR-extracted text from the meme
#         country      : cultural context (e.g. "India, China, Ireland")
#         language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

#     Returns:
#         dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
#     """
#     image_id = os.path.splitext(os.path.basename(image_path))[0]
#     prompt   = PROMPT.format(language=language, country=country)

#     # Append OCR transcription to the prompt
#     user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

#     messages = [{
#         "role": "user",
#         "content": [
#             {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
#             {"type": "text",  "text": user_message},
#         ],
#     }]

#     # Prepare inputs
#     text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
#                              padding=True, return_tensors="pt").to(DEVICE)

#     # Generate
#     with torch.no_grad():
#         out_ids = model.generate(**inputs, max_new_tokens=512,
#                                  do_sample=False, temperature=None, top_p=None)

#     result = processor.batch_decode(
#         [out_ids[0][len(inputs.input_ids[0]):]],
#         skip_special_tokens=True,
#     )[0].strip()

#     # Parse JSON response
#     try:
#         if "```json" in result:
#             json_str = result.split("```json")[1].split("```")[0].strip()
#         elif "```" in result:
#             json_str = result.split("```")[1].strip()
#         else:
#             start = result.find("{")
#             end   = result.rfind("}") + 1
#             json_str = result[start:end] if start >= 0 and end > start else result

#         parsed         = json.loads(json_str)
#         label          = int(parsed.get("label", 0))
#         classification = "misogyny" if label == 1 else "not-misogyny"
#         explanation    = parsed.get("reason", "")

#     except Exception as e:
#         print(f"  [WARN] JSON parse failed for {image_id}: {e}")
#         print(f"  Raw response: {result}")
#         classification = "misogyny" if ("misogyny" in result.lower()
#                                         and "not-misogyny" not in result.lower()
#                                         and "not misogyny" not in result.lower()) else "not-misogyny"
#         explanation = result or "No response."

#     return {
#         "image_id":       image_id,
#         "classification": classification,
#         "explanation":    explanation,
#         "full_response":  result,
#     }
def classify_image(image_path, transcription, country, language):

    image_id = os.path.splitext(os.path.basename(image_path))[0]

    prompt = PROMPT.format(language=language, country=country)

    # ❗ 不再使用图片
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    inputs = tokenizer(
        user_message,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    result = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # ---- JSON解析（保持你原来的）----
    try:
        start = result.find("{")
        end   = result.rfind("}") + 1
        json_str = result[start:end]

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"[WARN] JSON parse failed: {e}")
        classification = "not-misogyny"
        explanation = result

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 340/340 [00:07<00:00, 43.02it/s]


Model ready.



Classifying:   0%|▏                                                                   | 1/356 [00:00<01:48,  3.29meme/s]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


...

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying:  99%|█████████████████████████████████████████████████████████████████▍| 353/356 [20:01<00:12,  4.29s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)
[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [20:04<00:00,  3.38s/meme]


Results saved → Gemma-3-1B-it-abliterated/results_test_Tamil_India/models--mlabonne--gemma-3-1b-it-abliterated_tamil.json  |  Gemma-3-1B-it-abliterated/results_test_Tamil_India/models--mlabonne--gemma-3-1b-it-abliterated_tamil.txt

Summary:  Total=356  Misogyny=11  Not-misogyny=345  Errors=0


In [13]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
# from PIL import Image
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import AutoTokenizer, AutoModelForCausalLM
# from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------
# 🔧 1. 模型加载（替换整段）
# MODEL_NAME = "google/gemma-3-4b-it"
MODEL_NAME = "models--mlabonne--gemma-3-1b-it-abliterated"
# MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "China"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Gemma-3-1B-it-abliterated/results_test_Tamil_China/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
#     device_map="auto",
# )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
# 🔧 2. 删除这些内容（全部删掉）
# processor = AutoProcessor.from_pretrained(
#     MODEL_NAME,
#     min_pixels=256 * 28 * 28,
#     max_pixels=1280 * 28 * 28,
# )
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
# ✅ 改 prompt（很重要）
# PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
# Reason over BOTH the image and the transcription text together.

# Important Notes:
# - Be objective and precise.
# - Do not interpret humor or irony as neutral if it carries misogynistic meaning.

# Respond ONLY with this JSON (no markdown, no extra text):
# {{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
# label: 1 = misogyny, 0 = not-misogyny"""
PROMPT = """Given a {language} meme OCR transcription, classify whether it is misogynistic (1) or not (0) in {country} cultural context.

Note:
- You DO NOT see the image.
- Base your judgment only on the text.
- Be conservative: only label misogyny if clearly offensive toward women.

Respond ONLY JSON:
{{"label": 1, "reason": "..."}}
"""
# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------
# 🔧 3. 修改 classify_image（⚠️最重要）

# 👉 改成 纯文本推理
# def classify_image(image_path, transcription, country, language):
#     """
#     Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

#     Args:
#         image_path   : path to the meme image
#         transcription: OCR-extracted text from the meme
#         country      : cultural context (e.g. "India, China, Ireland")
#         language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

#     Returns:
#         dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
#     """
#     image_id = os.path.splitext(os.path.basename(image_path))[0]
#     prompt   = PROMPT.format(language=language, country=country)

#     # Append OCR transcription to the prompt
#     user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

#     messages = [{
#         "role": "user",
#         "content": [
#             {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
#             {"type": "text",  "text": user_message},
#         ],
#     }]

#     # Prepare inputs
#     text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
#                              padding=True, return_tensors="pt").to(DEVICE)

#     # Generate
#     with torch.no_grad():
#         out_ids = model.generate(**inputs, max_new_tokens=512,
#                                  do_sample=False, temperature=None, top_p=None)

#     result = processor.batch_decode(
#         [out_ids[0][len(inputs.input_ids[0]):]],
#         skip_special_tokens=True,
#     )[0].strip()

#     # Parse JSON response
#     try:
#         if "```json" in result:
#             json_str = result.split("```json")[1].split("```")[0].strip()
#         elif "```" in result:
#             json_str = result.split("```")[1].strip()
#         else:
#             start = result.find("{")
#             end   = result.rfind("}") + 1
#             json_str = result[start:end] if start >= 0 and end > start else result

#         parsed         = json.loads(json_str)
#         label          = int(parsed.get("label", 0))
#         classification = "misogyny" if label == 1 else "not-misogyny"
#         explanation    = parsed.get("reason", "")

#     except Exception as e:
#         print(f"  [WARN] JSON parse failed for {image_id}: {e}")
#         print(f"  Raw response: {result}")
#         classification = "misogyny" if ("misogyny" in result.lower()
#                                         and "not-misogyny" not in result.lower()
#                                         and "not misogyny" not in result.lower()) else "not-misogyny"
#         explanation = result or "No response."

#     return {
#         "image_id":       image_id,
#         "classification": classification,
#         "explanation":    explanation,
#         "full_response":  result,
#     }
def classify_image(image_path, transcription, country, language):

    image_id = os.path.splitext(os.path.basename(image_path))[0]

    prompt = PROMPT.format(language=language, country=country)

    # ❗ 不再使用图片
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    inputs = tokenizer(
        user_message,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    result = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # ---- JSON解析（保持你原来的）----
    try:
        start = result.find("{")
        end   = result.rfind("}") + 1
        json_str = result[start:end]

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"[WARN] JSON parse failed: {e}")
        classification = "not-misogyny"
        explanation = result

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 340/340 [00:08<00:00, 40.47it/s]


Model ready.



Classifying:   0%|▏                                                                   | 1/356 [00:00<01:49,  3.24meme/s]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


...

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)
[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying: 100%|█████████████████████████████████████████████████████████████████▊| 355/356 [18:58<00:04,  4.03s/meme]

[WARN] JSON parse failed: Extra data: line 2 column 1 (char 146)


Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [19:00<00:00,  3.20s/meme]


Results saved → Gemma-3-1B-it-abliterated/results_test_Tamil_China/models--mlabonne--gemma-3-1b-it-abliterated_tamil.json  |  Gemma-3-1B-it-abliterated/results_test_Tamil_China/models--mlabonne--gemma-3-1b-it-abliterated_tamil.txt

Summary:  Total=356  Misogyny=17  Not-misogyny=339  Errors=0


In [14]:
import os
import json
import pandas as pd
from tqdm import tqdm
import time
import torch
# from PIL import Image
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import AutoTokenizer, AutoModelForCausalLM
# from qwen_vl_utils import process_vision_info

# ============================================================
# CC-MMD 2026 — Starter Code
# Cross-Cultural Misogynistic Meme Detection
# ============================================================
#
# Install:
#   pip install git+https://github.com/huggingface/transformers accelerate
#   pip install qwen-vl-utils[decord]==0.0.8 pandas tqdm torch Pillow
# ============================================================

# ------------------------------------------------------------
# SETTINGS — edit these before running
# ------------------------------------------------------------
# 🔧 1. 模型加载（替换整段）
# MODEL_NAME = "google/gemma-3-4b-it"
MODEL_NAME = "models--mlabonne--gemma-3-1b-it-abliterated"
# MODEL_NAME = "models--Qwen--Qwen2.5-VL-7B-Instruct"
LANGUAGE   = "Tamil"    # Tamil | Malayalam | Chinese | English
COUNTRY    = "Ireland"    # cultural context for classification - India, China, Ireland

# IMAGE_DIR  = "sample/"   # folder containing meme images
IMAGE_DIR  = "dataset/Tamil/test/"   # folder containing meme images
# INPUT_CSV  = "sample/sample.csv" # MDMD dataset CSV
INPUT_CSV  = "dataset/Tamil/test/test.csv" # MDMD dataset CSV
OUTPUT_DIR = "Gemma-3-1B-it-abliterated/results_test_Tamil_Ireland/" # folder to save output files

# ------------------------------------------------------------
# LOAD MODEL (once at startup)
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on {DEVICE}...")

# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
#     device_map="auto",
# )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
# 🔧 2. 删除这些内容（全部删掉）
# processor = AutoProcessor.from_pretrained(
#     MODEL_NAME,
#     min_pixels=256 * 28 * 28,
#     max_pixels=1280 * 28 * 28,
# )
print("Model ready.\n")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
# ✅ 改 prompt（很重要）
# PROMPT = """Given a {language} meme image and its OCR transcription, classify whether the meme is misogynistic (1) or not (0) according to the cultural context of {country}.
# Reason over BOTH the image and the transcription text together.

# Important Notes:
# - Be objective and precise.
# - Do not interpret humor or irony as neutral if it carries misogynistic meaning.

# Respond ONLY with this JSON (no markdown, no extra text):
# {{"label": 1, "reason": "brief explanation referencing the image and transcription"}}
# label: 1 = misogyny, 0 = not-misogyny"""
PROMPT = """Given a {language} meme OCR transcription, classify whether it is misogynistic (1) or not (0) in {country} cultural context.

Note:
- You DO NOT see the image.
- Base your judgment only on the text.
- Be conservative: only label misogyny if clearly offensive toward women.

Respond ONLY JSON:
{{"label": 1, "reason": "..."}}
"""
# ------------------------------------------------------------
# CLASSIFY A SINGLE MEME
# ------------------------------------------------------------
# 🔧 3. 修改 classify_image（⚠️最重要）

# 👉 改成 纯文本推理
# def classify_image(image_path, transcription, country, language):
#     """
#     Classify a single meme image using HuggingFace Transformers (Qwen2.5-VL).

#     Args:
#         image_path   : path to the meme image
#         transcription: OCR-extracted text from the meme
#         country      : cultural context (e.g. "India, China, Ireland")
#         language     : transcription language (e.g. "Tamil, Malayalam, Chinese, English")

#     Returns:
#         dict with image_id, classification (misogyny / not-misogyny), explanation, full_response
#     """
#     image_id = os.path.splitext(os.path.basename(image_path))[0]
#     prompt   = PROMPT.format(language=language, country=country)

#     # Append OCR transcription to the prompt
#     user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

#     messages = [{
#         "role": "user",
#         "content": [
#             {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
#             {"type": "text",  "text": user_message},
#         ],
#     }]

#     # Prepare inputs
#     text         = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)
#     inputs       = processor(text=[text], images=image_inputs, videos=video_inputs,
#                              padding=True, return_tensors="pt").to(DEVICE)

#     # Generate
#     with torch.no_grad():
#         out_ids = model.generate(**inputs, max_new_tokens=512,
#                                  do_sample=False, temperature=None, top_p=None)

#     result = processor.batch_decode(
#         [out_ids[0][len(inputs.input_ids[0]):]],
#         skip_special_tokens=True,
#     )[0].strip()

#     # Parse JSON response
#     try:
#         if "```json" in result:
#             json_str = result.split("```json")[1].split("```")[0].strip()
#         elif "```" in result:
#             json_str = result.split("```")[1].strip()
#         else:
#             start = result.find("{")
#             end   = result.rfind("}") + 1
#             json_str = result[start:end] if start >= 0 and end > start else result

#         parsed         = json.loads(json_str)
#         label          = int(parsed.get("label", 0))
#         classification = "misogyny" if label == 1 else "not-misogyny"
#         explanation    = parsed.get("reason", "")

#     except Exception as e:
#         print(f"  [WARN] JSON parse failed for {image_id}: {e}")
#         print(f"  Raw response: {result}")
#         classification = "misogyny" if ("misogyny" in result.lower()
#                                         and "not-misogyny" not in result.lower()
#                                         and "not misogyny" not in result.lower()) else "not-misogyny"
#         explanation = result or "No response."

#     return {
#         "image_id":       image_id,
#         "classification": classification,
#         "explanation":    explanation,
#         "full_response":  result,
#     }
def classify_image(image_path, transcription, country, language):

    image_id = os.path.splitext(os.path.basename(image_path))[0]

    prompt = PROMPT.format(language=language, country=country)

    # ❗ 不再使用图片
    user_message = f"{prompt}\n\nOCR transcription:\n\"\"\"\n{transcription}\n\"\"\""

    inputs = tokenizer(
        user_message,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )

    result = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # ---- JSON解析（保持你原来的）----
    try:
        start = result.find("{")
        end   = result.rfind("}") + 1
        json_str = result[start:end]

        parsed         = json.loads(json_str)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("reason", "")

    except Exception as e:
        print(f"[WARN] JSON parse failed: {e}")
        classification = "not-misogyny"
        explanation = result

    return {
        "image_id":       image_id,
        "classification": classification,
        "explanation":    explanation,
        "full_response":  result,
    }

# ------------------------------------------------------------
# BATCH CLASSIFY
# ------------------------------------------------------------

def batch_classify(df, image_dir, country, language, output_base):
    """
    Classify all memes in a dataframe and save results to JSON and TXT.
    """
    os.makedirs(os.path.dirname(output_base) or ".", exist_ok=True)

    json_output = f"{output_base}.json"
    txt_output  = f"{output_base}.txt"

    with open(txt_output, "w") as f:
        f.write("CC-MMD 2026 — Misogyny Meme Classification Results\n")
        f.write("=" * 50 + "\n\n")

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classifying", unit="meme", colour="green"):
        image_path = os.path.join(image_dir, f"{row['image_id']}.jpg")

        if not os.path.exists(image_path):
            print(f"  [WARN] Image not found: {image_path} — skipping.")
            continue

        result = classify_image(
            image_path    = image_path,
            transcription = str(row["transcriptions"]),
            country       = country,
            language      = language,
        )
        results.append(result)

        # Write to TXT immediately
        with open(txt_output, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 50 + "\n\n")

    # Save JSON
    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Summary
    total        = len(results)
    misogyny     = sum(1 for r in results if r["classification"] == "misogyny")
    not_misogyny = sum(1 for r in results if r["classification"] == "not-misogyny")
    errors       = total - misogyny - not_misogyny

    print(f"\nResults saved → {json_output}  |  {txt_output}")
    print(f"\nSummary:  Total={total}  Misogyny={misogyny}  Not-misogyny={not_misogyny}  Errors={errors}")

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

if __name__ == "__main__":

    # --- Full dataset run (uncomment to use) ---
    df = pd.read_csv(INPUT_CSV)
    batch_classify(
        df          = df,
        image_dir   = IMAGE_DIR,
        country     = COUNTRY,
        language    = LANGUAGE,
        output_base = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.split('/')[-1]}_{LANGUAGE.lower()}"),
    )

Loading model on cuda...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 340/340 [00:06<00:00, 49.16it/s]


Model ready.



Classifying:   1%|▍                                                                   | 2/356 [00:03<08:29,  1.44s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


...

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying:  99%|█████████████████████████████████████████████████████████████████▍| 353/356 [21:13<00:12,  4.06s/meme]

[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)
[WARN] JSON parse failed: Expecting value: line 1 column 1 (char 0)


Classifying: 100%|██████████████████████████████████████████████████████████████████| 356/356 [21:15<00:00,  3.58s/meme]


Results saved → Gemma-3-1B-it-abliterated/results_test_Tamil_Ireland/models--mlabonne--gemma-3-1b-it-abliterated_tamil.json  |  Gemma-3-1B-it-abliterated/results_test_Tamil_Ireland/models--mlabonne--gemma-3-1b-it-abliterated_tamil.txt

Summary:  Total=356  Misogyny=12  Not-misogyny=344  Errors=0
